In [6]:
%load_ext autoreload
%autoreload 2
import sys, warnings
sys.path.append('..')
warnings.filterwarnings('ignore')

import pandas as pd
import pm4py
from pm4py.visualization.petri_net import visualizer as pn_visualizer
from python_analysis.functions import (
    read_log, distinct_values, show_trace_rows, unique_analysis, field_summary, traces_without,
    trace_share, start_end, event_code_legend, variants,
    activity_table, equivalence_classes, template, cover_table, assign_classes,
    metric, discover_and_evaluate, compare, combine_partitions, style_stats,
    discover_inductive_noise_twentieth, discover_ilp, discover_powl, discover_inductive_noise_tenth, direct_neighborhood, predecessor_times, successor_times, rejection_timeline
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
##Daten lesen
dom_log_base = read_log('../data/DomesticDeclarations.xes')
int_log_base = read_log('../data/InternationalDeclarations.xes')

dummy = int_log_base["case:Permit id"].ne(int_log_base["case:Permit ID"])
decl_and_permit = int_log_base[~dummy].copy()
international_only_decl = int_log_base[dummy].copy()
declarations_only = pd.concat(
    [dom_log_base.assign(**{"case:source": "domestic"}),
     international_only_decl.assign(**{"case:source": "international"})],
    ignore_index=True,
).sort_values(["case:concept:name", "time:timestamp"], kind="stable", ignore_index=True)

def split_by_trace_start(log, cutoff="2018-01-01"):
    events = log.loc[
        ~log["concept:name"].isin({"Start trip", "End trip"})
    ].copy()

    timestamps = pd.to_datetime(events["time:timestamp"], utc=True)
    starts = timestamps.groupby(events["case:concept:name"]).transform("min")
    before = starts.lt(pd.Timestamp(cutoff, tz="UTC"))

    return events.loc[before].copy(), events.loc[~before].copy()

logs = {
        #Rohdaten
        "domestic": dom_log_base,
        "international": int_log_base,
        #Aufteilung von international nach declaration only ohne Permit und den reinen Permit-enthaltenen Traces
        "int_decl_only": declarations_only,
        "int_permit": decl_and_permit,
        #Zus. ein Stack mit allen Traces nur aus Declaration (ohne Trips)
        "all_decl_only": pd.concat([dom_log_base, declarations_only[~declarations_only["concept:name"].isin({"Start trip", "End trip"})]], ignore_index=True),
        #Trips rausnehmen, weil unabhängig (betrifft nur die internationalen Traces)
        "int_without_trips": int_log_base[~int_log_base["concept:name"].isin({"Start trip", "End trip"})],
        "int_decl_only_without_trips": declarations_only[~declarations_only["concept:name"].isin({"Start trip", "End trip"})],
        "int_permit_without_trips": decl_and_permit[~decl_and_permit["concept:name"].isin({"Start trip", "End trip"})],

        #"domestic_without_D20": dom_log_base[~dom_log_base["concept:name"].eq("Declaration SAVED by EMPLOYEE")],

        #"int_without_trips_or_D20": int_log_base[~int_log_base["concept:name"].isin({"Start trip", "End trip", "Declaration SAVED by EMPLOYEE"})],
        #"int_only_decl": international_only_decl,
        #"int_only_decl_without_trips": international_only_decl[~international_only_decl["concept:name"].isin({"Start trip", "End trip"})],
        #"int_only_decl_without_trips_or_D20": international_only_decl[~international_only_decl["concept:name"].isin({"Start trip", "End trip", "Declaration SAVED by EMPLOYEE"})],


}

# 2017/2018 Split
logs["domestic_before_2018"], logs["domestic_from_2018"] = (
    split_by_trace_start(logs["domestic"])
)

logs["int_without_trips_before_2018"], logs["int_without_trips_from_2018"] = (
    split_by_trace_start(logs["int_without_trips"])
)

logs["int_decl_only_without_trips_before_2018"], logs["int_decl_only_without_trips_from_2018"] = (
    split_by_trace_start(logs["int_decl_only_without_trips"])
)

logs["int_permit_without_trips_before_2018"], logs["int_permit_without_trips_from_2018"] = (
    split_by_trace_start(logs["int_permit_without_trips"])
)

def add_2018_marker(l) :
    source = l.copy()
    source["time:timestamp"] = pd.to_datetime(
        source["time:timestamp"], utc=True
    )
    source["synthetic"] = False

    # Je Trace ein Marker; nur die Trace-Attribute übernehmen.
    case_columns = [c for c in source.columns if c.startswith("case:")]
    markers = (
        source[case_columns]
        .drop_duplicates(subset="case:concept:name")
        .reindex(columns=source.columns)
        .copy()
    )

    markers["concept:name"] = "Jahresgrenze 2018"
    markers["time:timestamp"] = pd.Timestamp("2018-01-01", tz="UTC")
    markers["synthetic"] = True

    if "org:resource" in markers.columns:
        markers["org:resource"] = "SYSTEM"
    if "org:role" in markers.columns:
        markers["org:role"] = "UNDEFINED"
    return (
        pd.concat([source, markers], ignore_index=True)
        .sort_values(
            ["case:concept:name", "time:timestamp", "synthetic"],
            ascending=[True, True, False],
            kind="stable",
            ignore_index=True,
        )
    )

logs["int_without_trips_with_year_marker"] = add_2018_marker(logs["int_without_trips"])
logs["dom_with_year_marker"] = add_2018_marker(logs["domestic"])

parsing log, completed traces :: 100%|██████████| 6449/6449 [00:01<00:00, 4238.75it/s]


In [8]:
MINERS = {
    #pm4py
    "inductivepn": pm4py.discover_petri_net_inductive,
    "alphapn": pm4py.discover_petri_net_alpha,
    "alphapluspn": pm4py.discover_petri_net_alpha_plus,
    "heuristicpn": pm4py.discover_petri_net_heuristics,
    "inductivebpmn": pm4py.discover_bpmn_inductive,
    #functions (auch pm4py, aber noch mit Mini-Wrapper für dieses invoke-Ding da unten)
    "inductive_noise10pn": discover_inductive_noise_tenth,
    "inductive_noise20pn": discover_inductive_noise_twentieth,
    "ilppn": discover_ilp,
    "powl": discover_powl,
}

models, stats = {}, {}
for name, log in logs.items():
    for miner, discover in MINERS.items():
        models[(name, miner)], stats[(name, miner)] = discover_and_evaluate(name, miner, discover, log)
table = pd.DataFrame(stats)
table.columns.names = ["Log", "Verfahren"]
display(style_stats(compare(table)))
display(style_stats(table))

domestic_inductivepn_model.pkl: aus Cache geladen
domestic_inductivepn_stats.pkl: 19 aus Cache, neu berechnet: –
domestic_alphapn_model.pkl: aus Cache geladen
domestic_alphapn_stats.pkl: 19 aus Cache, neu berechnet: –
domestic_alphapluspn_model.pkl: aus Cache geladen
domestic_alphapluspn_stats.pkl: 19 aus Cache, neu berechnet: –
domestic_heuristicpn_model.pkl: aus Cache geladen
domestic_heuristicpn_stats.pkl: 19 aus Cache, neu berechnet: –
domestic_inductivebpmn_model.pkl: aus Cache geladen
domestic_inductivebpmn_stats.pkl: 19 aus Cache, neu berechnet: –
domestic_inductive_noise10pn_model.pkl: aus Cache geladen
domestic_inductive_noise10pn_stats.pkl: 19 aus Cache, neu berechnet: –
domestic_inductive_noise20pn_model.pkl: aus Cache geladen
domestic_inductive_noise20pn_stats.pkl: 19 aus Cache, neu berechnet: –
domestic_ilppn_model.pkl: aus Cache geladen
domestic_ilppn_stats.pkl: 19 aus Cache, neu berechnet: –
domestic_powl_model.pkl: aus Cache geladen
domestic_powl_stats.pkl: 19 aus Cache

In [9]:
# Log-Gruppen, die jeweils zusammen alle Traces abdecken: raw vs. vorverarbeitet vs. ohne Trips vs. drei Prozesse getrennt
print("[oT] = ohne Trips")
PARTITIONS = {      "raw": ["domestic", "international"],
    "v1 (decl / declAndPermits)": ["all_decl_only", "int_permit"],
    "v2 (dom / int_decl_only / int_permits)": ["domestic", "int_decl_only", "int_permit"],
    "raw [oT]": ["domestic", "int_without_trips"],
    "v1 Duo [oT]": ["all_decl_only", "int_permit_without_trips"],
    "v2 Triple [oT]": ["domestic", "int_decl_only_without_trips", "int_permit_without_trips"],
    "2017/later Split [oT]": ["domestic_before_2018", "domestic_from_2018", "int_without_trips_before_2018", "int_without_trips_from_2018"],
    "2017-Triple-Split [oT]":["domestic_before_2018", "int_decl_only_without_trips_before_2018", "int_permit_without_trips_before_2018"],
    "year_event [oT]": ["dom_with_year_marker", "int_without_trips_with_year_marker"]
    #"getrennt ohne Trips": ["domestic", "decl_and_permit_without_trips", "international_only_decl_without_trips"],
    #"getrennt ohne Trips, ohne SAVE": ["domestic_without_D20", "decl_and_permit_without_trips_or_D20", "international_only_decl_without_trips_or_D20"],
}
combined = combine_partitions(table, logs, PARTITIONS)
display(style_stats(combined[["heuristicpn"]]))
display(style_stats(combined))


[oT] = ohne Trips


In [10]:
##Datenfelder je Trace / Event
field_summary(logs).style \
    .set_caption("Datenfelder je Trace / Event") \
    .set_properties(**{"text-align": "left"})

,Log,Eventfelder,Tracefelder
0,domestic,"['id', 'org:resource', 'concept:name', 'time:timestamp', 'org:role']","['case:id', 'case:concept:name', 'case:BudgetNumber', 'case:DeclarationNumber', 'case:Amount']"
1,international,"['id', 'org:resource', 'concept:name', 'time:timestamp', 'org:role']","['case:Permit travel permit number', 'case:DeclarationNumber', 'case:Amount', 'case:RequestedAmount', 'case:Permit TaskNumber', 'case:Permit BudgetNumber', 'case:OriginalAmount', 'case:Permit ProjectNumber', 'case:concept:name', 'case:Permit OrganizationalEntity', 'case:travel permit number', 'case:Permit RequestedBudget', 'case:id', 'case:Permit ID', 'case:Permit id', 'case:BudgetNumber', 'case:Permit ActivityNumber', 'case:AdjustedAmount']"
2,int_decl_only,"['id', 'org:resource', 'concept:name', 'time:timestamp', 'org:role']","['case:id', 'case:concept:name', 'case:BudgetNumber', 'case:DeclarationNumber', 'case:Amount', 'case:source', 'case:Permit travel permit number', 'case:RequestedAmount', 'case:Permit TaskNumber', 'case:Permit BudgetNumber', 'case:OriginalAmount', 'case:Permit ProjectNumber', 'case:Permit OrganizationalEntity', 'case:travel permit number', 'case:Permit RequestedBudget', 'case:Permit ID', 'case:Permit id', 'case:Permit ActivityNumber', 'case:AdjustedAmount']"
3,int_permit,"['id', 'org:resource', 'concept:name', 'time:timestamp', 'org:role']","['case:Permit travel permit number', 'case:DeclarationNumber', 'case:Amount', 'case:RequestedAmount', 'case:Permit TaskNumber', 'case:Permit BudgetNumber', 'case:OriginalAmount', 'case:Permit ProjectNumber', 'case:concept:name', 'case:Permit OrganizationalEntity', 'case:travel permit number', 'case:Permit RequestedBudget', 'case:id', 'case:Permit ID', 'case:Permit id', 'case:BudgetNumber', 'case:Permit ActivityNumber', 'case:AdjustedAmount']"
4,all_decl_only,"['id', 'org:resource', 'concept:name', 'time:timestamp', 'org:role']","['case:id', 'case:concept:name', 'case:BudgetNumber', 'case:DeclarationNumber', 'case:Amount', 'case:source', 'case:Permit travel permit number', 'case:RequestedAmount', 'case:Permit TaskNumber', 'case:Permit BudgetNumber', 'case:OriginalAmount', 'case:Permit ProjectNumber', 'case:Permit OrganizationalEntity', 'case:travel permit number', 'case:Permit RequestedBudget', 'case:Permit ID', 'case:Permit id', 'case:Permit ActivityNumber', 'case:AdjustedAmount']"
5,int_without_trips,"['id', 'org:resource', 'concept:name', 'time:timestamp', 'org:role']","['case:Permit travel permit number', 'case:DeclarationNumber', 'case:Amount', 'case:RequestedAmount', 'case:Permit TaskNumber', 'case:Permit BudgetNumber', 'case:OriginalAmount', 'case:Permit ProjectNumber', 'case:concept:name', 'case:Permit OrganizationalEntity', 'case:travel permit number', 'case:Permit RequestedBudget', 'case:id', 'case:Permit ID', 'case:Permit id', 'case:BudgetNumber', 'case:Permit ActivityNumber', 'case:AdjustedAmount']"
6,int_decl_only_without_trips,"['id', 'org:resource', 'concept:name', 'time:timestamp', 'org:role']","['case:id', 'case:concept:name', 'case:BudgetNumber', 'case:DeclarationNumber', 'case:Amount', 'case:source', 'case:Permit travel permit number', 'case:RequestedAmount', 'case:Permit TaskNumber', 'case:Permit BudgetNumber', 'case:OriginalAmount', 'case:Permit ProjectNumber', 'case:Permit OrganizationalEntity', 'case:travel permit number', 'case:Permit RequestedBudget', 'case:Permit ID', 'case:Permit id', 'case:Permit ActivityNumber', 'case:AdjustedAmount']"
7,int_permit_without_trips,"['id', 'org:resource', 'concept:name', 'time:timestamp', 'org:role']","['case:Permit travel permit number', 'case:DeclarationNumber', 'case:Amount', 'case:RequestedAmount', 'case:Permit TaskNumber', 'case:Permit BudgetNumber', 'case:OriginalAmount', 'case:Permit ProjectNumber', 'case:concept:name', 'case:Permit OrganizationalEntity', 'case:travel permit number', 'case:Permit RequestedBudget', 'case:id', 'case:Permit ID', 'case:Permit id', 'case:BudgetNumber', 'case:Permit ActivityNumber', 

In [11]:
print("\nDomestic Log (10500):")
for f in dom_log_base.columns:
    print(str(f) + ' : '+ str(len(distinct_values(dom_log_base, f))))

print("\nInternational Log (6449):")
for f in int_log_base.columns:
    print(str(f) + ' : '+ str(len(distinct_values(int_log_base, f))))


Domestic Log (10500):
id : 56437
org:resource : 2
concept:name : 17
time:timestamp : 45403
org:role : 7
case:id : 10500
case:concept:name : 10500
case:BudgetNumber : 1
case:DeclarationNumber : 10049
case:Amount : 8326

International Log (6449):
id : 69073
org:resource : 2
concept:name : 34
time:timestamp : 51270
org:role : 8
case:Permit travel permit number : 5596
case:DeclarationNumber : 6190
case:Amount : 6100
case:RequestedAmount : 6100
case:Permit TaskNumber : 6
case:Permit BudgetNumber : 207
case:OriginalAmount : 6100
case:Permit ProjectNumber : 825
case:concept:name : 6449
case:Permit OrganizationalEntity : 27
case:travel permit number : 6033
case:Permit RequestedBudget : 5259
case:id : 6449
case:Permit ID : 6028
case:Permit id : 5608
case:BudgetNumber : 719
case:Permit ActivityNumber : 145
case:AdjustedAmount : 6101


In [12]:
unique_analysis(int_log_base, ["case:OriginalAmount", "case:AdjustedAmount", "case:RequestedAmount", "case:Amount"])

,case:OriginalAmount,case:AdjustedAmount,case:RequestedAmount,case:Amount,traces
0,0.000000,0.000000,0.000000,0.000000,2251
1,45.654385,45.654385,45.654385,45.654385,73
2,64.473928,64.473928,64.473928,64.473928,72
3,49.536371,49.536371,49.536371,49.536371,58
4,35.122357,35.122357,35.122357,35.122357,52


True     6448
False       1
Name: count, dtype: int64


,case:id,case:OriginalAmount,case:AdjustedAmount,case:RequestedAmount,case:Amount
0,declaration 54434,0.0,100.494292,0.0,0.0


['declaration 54434']

In [45]:
duplicates = dom_log_base["id"].value_counts().loc[lambda counts: counts > 1]
x = duplicates.sum()

print(f"Betroffene Events: {x}")
print(f"Zusätzliche Vorkommen (x - {len(duplicates)}): {x - len(duplicates)}")

Betroffene Events: 0
Zusätzliche Vorkommen (x - 0): 0


In [46]:
duplicates = int_log_base["id"].value_counts().loc[lambda counts: counts > 1]
x = duplicates.sum()

print(f"Betroffene Events: {x}")
print(f"Zusätzliche Vorkommen (x - {len(duplicates)}): {x - len(duplicates)}")

Betroffene Events: 4558
Zusätzliche Vorkommen (x - 1480): 3078


In [48]:
id_analysis = int_log_base.groupby("id").agg(
    events=("id", "size"),
    distinct_timestamps=("time:timestamp", lambda s: s.nunique(dropna=False)),
    distinct_concept_name=("concept:name", lambda s: s.nunique(dropna=False)),
)
duplicates = id_analysis.loc[id_analysis["events"] > 1]
conflicts = duplicates.loc[duplicates["distinct_timestamps"] > 1]
x = duplicates["events"].sum()

print(f"Mehrfach vorkommende IDs: {len(duplicates)}")
print(f"Betroffene Events: {x}")
print(f"Zusätzliche Vorkommen: {x - len(duplicates)}")
print(f"IDs mit identischem Timestamp: {len(duplicates) - len(conflicts)}")
print(f"IDs mit unterschiedlichen Timestamps: {len(conflicts)}")

print("Konzeptname")
print(duplicates.loc[duplicates["distinct_concept_name"] > 1])

display(conflicts)

Mehrfach vorkommende IDs: 1480
Betroffene Events: 4558
Zusätzliche Vorkommen: 3078
IDs mit identischem Timestamp: 1480
IDs mit unterschiedlichen Timestamps: 0
Konzeptname
Empty DataFrame
Columns: [events, distinct_timestamps, distinct_concept_name]
Index: []


,events,distinct_timestamps,distinct_concept_name
id,,,


In [13]:
#Permit Dummy Analyse 448 Traces
problematic = set(unique_analysis(int_log_base, ["case:Permit id", "case:Permit ID"]))
without_permit = traces_without(int_log_base, "Permit")

print("Traces ohne Permit-Event:", len(without_permit))
print("problematic_traces:", len(problematic))
print("Gleiche Menge:", without_permit == problematic)
print("Nur ohne Permit:", sorted(without_permit - problematic))
print("Nur in problematic_traces:", sorted(problematic - without_permit))

,case:Permit id,case:Permit ID,traces
0,travel permit 73562,travel permit 73562,170
1,travel permit 53343,travel permit 53343,140
2,travel permit 32577,travel permit 32577,115
3,travel permit 45618,travel permit 45618,98
4,travel permit 64232,travel permit 64232,84


True     6001
False     448
Name: count, dtype: int64


,case:id,case:Permit id,case:Permit ID
0,declaration 143637,travel permit 423,travel permit 143640
1,declaration 143644,travel permit 423,travel permit 143647
2,declaration 143578,travel permit 423,travel permit 143581
3,declaration 143585,travel permit 423,travel permit 143588
4,declaration 143657,travel permit 423,travel permit 143660
...,...,...,...
443,declaration 147452,travel permit 423,travel permit 147455
444,declaration 147470,travel permit 423,travel permit 147473
445,declaration 147487,travel permit 423,travel permit 147490
446,declaration 147507,travel permit 423,travel permit 147509


Traces ohne Permit-Event: 448
problematic_traces: 448
Gleiche Menge: True
Nur ohne Permit: []
Nur in problematic_traces: []


In [14]:
# Erst alle Aktivitäten anzeigen
event_types = pd.concat([pd.Series(dom_log_base["concept:name"].values), pd.Series(int_log_base["concept:name"].values)]).drop_duplicates().reset_index(drop=True).sort_values(ignore_index=True)
display(event_types)

# Dann Relationen finden: Wörter mit identischer Aktivitätsmenge bilden eine Äquivalenzklasse
acts = activity_table({"domestic": dom_log_base, "international": int_log_base})
eq = equivalence_classes(acts)
display(eq)

# Masken/Templates je Klasse definieren
eq["template"] = eq["members"].apply(template)
display(eq)

# Unnötige Masken entfernen, weil sie alles/nichts matchen
filtered_groups = eq[~eq["template"].str.fullmatch(r"[\s%]*", na=True)]
display(filtered_groups)

# Gruppen an Masken definieren, die genau jede Aktivität genau einmal abdecken
covers = cover_table(filtered_groups, set(acts.index))
display(covers)

acts["entity_classes"] = assign_classes(acts, filtered_groups, covers.at[1, "ids"])
acts["activity_classes"] = assign_classes(acts, filtered_groups, covers.at[11, "ids"])

0         Declaration APPROVED by ADMINISTRATION
1           Declaration APPROVED by BUDGET OWNER
2           Declaration APPROVED by PRE_APPROVER
3             Declaration APPROVED by SUPERVISOR
4         Declaration FINAL_APPROVED by DIRECTOR
5       Declaration FINAL_APPROVED by SUPERVISOR
6     Declaration FOR_APPROVAL by ADMINISTRATION
7       Declaration FOR_APPROVAL by PRE_APPROVER
8         Declaration FOR_APPROVAL by SUPERVISOR
9         Declaration REJECTED by ADMINISTRATION
10          Declaration REJECTED by BUDGET OWNER
11              Declaration REJECTED by DIRECTOR
12              Declaration REJECTED by EMPLOYEE
13               Declaration REJECTED by MISSING
14          Declaration REJECTED by PRE_APPROVER
15            Declaration REJECTED by SUPERVISOR
16                 Declaration SAVED by EMPLOYEE
17             Declaration SUBMITTED by EMPLOYEE
18                                      End trip
19                               Payment Handled
20             Permi

,words,n,domestic,international,members
0,MEMBER STAFF,34,36353,59347,"[Declaration APPROVED by ADMINISTRATION, Decla..."
1,BY,32,36353,46449,"[Declaration APPROVED by ADMINISTRATION, Decla..."
2,DECLARATION,18,36353,25890,"[Declaration APPROVED by ADMINISTRATION, Decla..."
3,PERMIT,14,0,20559,"[Permit APPROVED by ADMINISTRATION, Permit APP..."
4,REJECTED,14,2846,4192,"[Declaration REJECTED by ADMINISTRATION, Decla..."
5,APPROVED,8,11707,15516,"[Declaration APPROVED by ADMINISTRATION, Decla..."
6,EMPLOYEE,7,13031,29338,"[Declaration REJECTED by EMPLOYEE, Declaration..."
7,SUPERVISOR,7,10425,12535,"[Declaration APPROVED by SUPERVISOR, Declarati..."
8,ADMINISTRATION,5,9155,11508,"[Declaration APPROVED by ADMINISTRATION, Decla..."
9,PRE_APPROVER,5,772,1255,"[Declaration APPROVED by PRE_APPROVER, Declara..."


,words,n,domestic,international,members,template
0,MEMBER STAFF,34,36353,59347,"[Declaration APPROVED by ADMINISTRATION, Decla...",%
1,BY,32,36353,46449,"[Declaration APPROVED by ADMINISTRATION, Decla...",% by %
2,DECLARATION,18,36353,25890,"[Declaration APPROVED by ADMINISTRATION, Decla...",Declaration % by %
3,PERMIT,14,0,20559,"[Permit APPROVED by ADMINISTRATION, Permit APP...",Permit % by %
4,REJECTED,14,2846,4192,"[Declaration REJECTED by ADMINISTRATION, Decla...",% REJECTED by %
5,APPROVED,8,11707,15516,"[Declaration APPROVED by ADMINISTRATION, Decla...",% APPROVED by %
6,EMPLOYEE,7,13031,29338,"[Declaration REJECTED by EMPLOYEE, Declaration...",%
7,SUPERVISOR,7,10425,12535,"[Declaration APPROVED by SUPERVISOR, Declarati...",% by SUPERVISOR
8,ADMINISTRATION,5,9155,11508,"[Declaration APPROVED by ADMINISTRATION, Decla...",% by ADMINISTRATION
9,PRE_APPROVER,5,772,1255,"[Declaration APPROVED by PRE_APPROVER, Declara...",% by PRE_APPROVER


,words,n,domestic,international,members,template
1,BY,32,36353,46449,"[Declaration APPROVED by ADMINISTRATION, Decla...",% by %
2,DECLARATION,18,36353,25890,"[Declaration APPROVED by ADMINISTRATION, Decla...",Declaration % by %
3,PERMIT,14,0,20559,"[Permit APPROVED by ADMINISTRATION, Permit APP...",Permit % by %
4,REJECTED,14,2846,4192,"[Declaration REJECTED by ADMINISTRATION, Decla...",% REJECTED by %
5,APPROVED,8,11707,15516,"[Declaration APPROVED by ADMINISTRATION, Decla...",% APPROVED by %
7,SUPERVISOR,7,10425,12535,"[Declaration APPROVED by SUPERVISOR, Declarati...",% by SUPERVISOR
8,ADMINISTRATION,5,9155,11508,"[Declaration APPROVED by ADMINISTRATION, Decla...",% by ADMINISTRATION
9,PRE_APPROVER,5,772,1255,"[Declaration APPROVED by PRE_APPROVER, Declara...",% by PRE_APPROVER
10,BUDGET OWNER,4,2879,3668,"[Declaration APPROVED by BUDGET OWNER, Declara...",% by BUDGET OWNER
11,DIRECTOR,4,0,897,"[Declaration FINAL_APPROVED by DIRECTOR, Decla...",% by DIRECTOR


,n_groups,groups,ids
0,4,"[BY, PAYMENT, TRIP, REMINDER SEND]","[1, 16, 18, 21]"
1,5,"[DECLARATION, PERMIT, PAYMENT, TRIP, REMINDER ...","[2, 3, 16, 18, 21]"
2,5,"[BY, TRIP, HANDLED, REMINDER SEND, REQUEST]","[1, 18, 20, 21, 22]"
3,5,"[BY, PAYMENT, END, REMINDER SEND, START]","[1, 16, 19, 21, 24]"
4,6,"[DECLARATION, PERMIT, TRIP, HANDLED, REMINDER ...","[2, 3, 18, 20, 21, 22]"
5,6,"[DECLARATION, PERMIT, PAYMENT, END, REMINDER S...","[2, 3, 16, 19, 21, 24]"
6,6,"[BY, END, HANDLED, REMINDER SEND, REQUEST, START]","[1, 19, 20, 21, 22, 24]"
7,7,"[DECLARATION, PERMIT, END, HANDLED, REMINDER S...","[2, 3, 19, 20, 21, 22, 24]"
8,9,"[REJECTED, APPROVED, FINAL_APPROVED, FOR_APPRO...","[4, 5, 12, 13, 16, 17, 18, 21, 23]"
9,10,"[REJECTED, APPROVED, FINAL_APPROVED, FOR_APPRO...","[4, 5, 12, 13, 17, 18, 20, 21, 22, 23]"


Menschliche Bewertung:
| index | Ausschlusskriterium? | Bewertung |
|-------|----------------------|-----------|
| 0     |Ja, durch "By" -> Permit/Declaration fälschlich zusammengefasst||
| 1     || Sehr gut, genau 5 Gruppen für 5 "menschlich erkennbare" [Entitätsklassen]|
| 2     |Ja, durch "By", s. (1)||
| 3     |Ja, durch "By", s. (1)||
| 4     |Ja, Aktivitäten und Entitäten werden gemischt "End, Payment, Start, etc."||
| 5     ||Ist okay, im Prinzip Gruppe (1), aber mit extra Aufteilung von "%Payment%-Aktivitäten"|
| 6     |Ja, durch "By", s. (1)||
| 7     |s. (4)||
| 8     |Ja, ähnlich zu (4) werden Aktivitäten und Entitäten gemischt (Trip, Reminder, Payment + Submitted, Approval...)||
| 9     |Ja, ebenfalls noch Entitäten (Trip) und Aktivitäten||
| 10    ||Fast gültig, es verbleiben noch eine zwei, die eineindeutige Aktivität (Reminder Send) und Payment, sonst alles abgedeckt|
| 11    ||Teilt wie bei (1) zu (5) nur das Payment auf, wodurch aber nur noch eine Entität mit eindeutiger Aktivität verbleibt -> Perfekt als [Aktivitätenklassen]|


Es verbleibt noch ein Problem, was nicht direkt ersichtlich ist: Durch die vorhandenen Matcher entsteht bei der Betrachtung der Events in ihren Klassen ein Informationsverlust bei den Aktivitäten durch den Matcher "%APPROVED %", deswegen darf beim Aufbau der Klassen nicht einfach eine Transformation zu Buchstaben erfolgen, sondern der Unterstrich beibehalten werden.

Die entstehenden Gruppierungen zeigen außerdem direkt auf, dass eindeutige Unterscheidungen nach Akteuren nicht zu einer allgemeingültigen bijektiven Abbildung führen und demnach nicht **eigenständig** geeignet für isolierte Analysen sind.

In [15]:
acts.groupby(["entity_classes", "activity_classes"])[["domestic", "international"]].sum()

domestic  international
entity_classes     activity_classes                                      
% Payment %        Payment Handled                   10044           6187
                   Request Payment                   10040           6183
% trip             End trip                              0           6449
                   Start trip                            0           6449
Declaration % by % % APPROVED by %                   11707           7739
                   % FINAL_APPROVED by %             10131           6291
                   % REJECTED by %                    2846           3686
                   % SUBMITTED by EMPLOYEE           11531           8099
                   Declaration FOR_APPROVAL by %         3              0
                   Declaration SAVED by EMPLOYEE       135             75
Permit % by %      % APPROVED by %                       0           7777
                   % FINAL_APPROVED by %                 0           6021
                   % REJECTED by %                       0            506
                   % SUBMITTED by EMPLOYEE               0           6255
Send Reminder      Send Reminder                         0            434

In [16]:
trace_share(logs, "concept:name")

,domestic,international,int_decl_only,int_permit,all_decl_only,int_without_trips,int_decl_only_without_trips,int_permit_without_trips,domestic_before_2018,domestic_from_2018,int_without_trips_before_2018,int_without_trips_from_2018,int_decl_only_without_trips_before_2018,int_decl_only_without_trips_from_2018,int_permit_without_trips_before_2018,int_permit_without_trips_from_2018,int_without_trips_with_year_marker,dom_with_year_marker
concept:name,,,,,,,,,,,,,,,,,,
Declaration APPROVED by ADMINISTRATION,7983/10500 [76.03%],4919/6449 [76.28%],8234/10948 [75.21%],4668/6001 [77.79%],8234/10948 [75.21%],4919/6449 [76.28%],8234/10948 [75.21%],4668/6001 [77.79%],18/2240 [0.80%],7965/8260 [96.43%],113/1450 [7.79%],4806/4999 [96.14%],18/2422 [0.74%],8216/8526 [96.36%],113/1268 [8.91%],4555/4733 [96.24%],4919/6449 [76.28%],7983/10500 [76.03%]
Declaration APPROVED by BUDGET OWNER,2807/10500 [26.73%],1817/6449 [28.17%],2924/10948 [26.71%],1700/6001 [28.33%],2924/10948 [26.71%],1817/6449 [28.17%],2924/10948 [26.71%],1700/6001 [28.33%],4/2240 [0.18%],2803/8260 [33.93%],27/1450 [1.86%],1790/4999 [35.81%],4/2422 [0.17%],2920/8526 [34.25%],27/1268 [2.13%],1673/4733 [35.35%],1817/6449 [28.17%],2807/10500 [26.73%]
Declaration APPROVED by PRE_APPROVER,666/10500 [6.34%],558/6449 [8.65%],712/10948 [6.50%],512/6001 [8.53%],712/10948 [6.50%],558/6449 [8.65%],712/10948 [6.50%],512/6001 [8.53%],666/2240 [29.73%],,558/1450 [38.48%],,712/2422 [29.40%],,512/1268 [40.38%],,558/6449 [8.65%],666/10500 [6.34%]
Declaration APPROVED by SUPERVISOR,,246/6449 [3.81%],34/10948 [0.31%],212/6001 [3.53%],34/10948 [0.31%],246/6449 [3.81%],34/10948 [0.31%],212/6001 [3.53%],,,99/1450 [6.83%],147/4999 [2.94%],16/2422 [0.66%],18/8526 [0.21%],83/1268 [6.55%],129/4733 [2.73%],246/6449 [3.81%],
Declaration FINAL_APPROVED by DIRECTOR,,244/6449 [3.78%],34/10948 [0.31%],210/6001 [3.50%],34/10948 [0.31%],244/6449 [3.78%],34/10948 [0.31%],210/6001 [3.50%],,,98/1450 [6.76%],146/4999 [2.92%],16/2422 [0.66%],18/8526 [0.21%],82/1268 [6.47%],128/4733 [2.70%],244/6449 [3.78%],
Declaration FINAL_APPROVED by SUPERVISOR,10070/10500 [95.90%],5961/6449 [92.43%],10463/10948 [95.57%],5568/6001 [92.78%],10463/10948 [95.57%],5961/6449 [92.43%],10463/10948 [95.57%],5568/6001 [92.78%],2167/2240 [96.74%],7903/8260 [95.68%],1320/1450 [91.03%],4641/4999 [92.84%],2328/2422 [96.12%],8135/8526 [95.41%],1159/1268 [91.40%],4409/4733 [93.15%],5961/6449 [92.43%],10070/10500 [95.90%]
Declaration FOR_APPROVAL by ADMINISTRATION,1/10500 [0.01%],,1/10948 [0.01%],,1/10948 [0.01%],,1/10948 [0.01%],,,1/8260 [0.01%],,,,1/8526 [0.01%],,,,1/10500 [0.01%]
Declaration FOR_APPROVAL by PRE_APPROVER,1/10500 [0.01%],,1/10948 [0.01%],,1/10948 [0.01%],,1/10948 [0.01%],,1/2240 [0.04%],,,,1/2422 [0.04%],,,,,1/10500 [0.01%]
Declaration FOR_APPROVAL by SUPERVISOR,1/10500 [0.01%],,1/10948 [0.01%],,1/10948 [0.01%],,1/10948 [0.01%],,1/2240 [0.04%],,,,1/2422 [0.04%],,,,,1/10500 [0.01%]


In [17]:
display(event_code_legend(logs))

with pd.option_context("display.max_colwidth", None):
    for name, log in logs.items():
        table = variants(log)
        print("" + "=" * 70)
        print(f"\n{name}: {len(table)} Abläufe")
        display(table)

,concept:name,Code
0,Declaration APPROVED by ADMINISTRATION,D36
1,Declaration APPROVED by BUDGET OWNER,D31
2,Declaration APPROVED by PRE_APPROVER,D32
3,Declaration APPROVED by SUPERVISOR,D35
4,Declaration FINAL_APPROVED by DIRECTOR,D5✅7
5,Declaration FINAL_APPROVED by SUPERVISOR,D5✅5
6,Declaration FOR_APPROVAL by ADMINISTRATION,D46
7,Declaration FOR_APPROVAL by PRE_APPROVER,D42
8,Declaration FOR_APPROVAL by SUPERVISOR,D45
9,Declaration REJECTED by ADMINISTRATION,D5❌6



domestic: 99 Abläufe


,Ablauf,Traces,Anteil,Median (Tage),Durchschnitt (Tage)
0,D10 > D36 > D5✅5 > 🪙❓ > 🪙❗,4618,4618/10500 [43.98%],7.036163,9.377247
1,D10 > D36 > D31 > D5✅5 > 🪙❓ > 🪙❗,2473,2473/10500 [23.55%],10.232593,12.550129
2,D10 > D5✅5 > 🪙❓ > 🪙❗,1392,1392/10500 [13.26%],6.311244,9.891926
3,D10 > D32 > D5✅5 > 🪙❓ > 🪙❗,575,575/10500 [5.48%],7.310475,9.966244
4,D10 > D5❌6 > D5❌0 > D10 > D36 > D5✅5 > 🪙❓ > 🪙❗,345,345/10500 [3.29%],11.104410,18.572796
...,...,...,...,...,...
94,D10 > D5❌2 > D5❌0 > D10 > D32 > D5✅5 > D5❌_,1,1/10500 [0.01%],47.078252,47.078252
95,D10 > D5✅5 > D5❌_ > D10 > D5❌5 > D5❌0 > D10 > D5✅5 > 🪙❓ > 🪙❗,1,1/10500 [0.01%],16.774051,16.774051
96,D20 > 🪙❓ > 🪙❗,1,1/10500 [0.01%],20.088113,20.088113
97,D10 > D42 > D5❌_,1,1/10500 [0.01%],4.134444,4.134444



international: 753 Abläufe


,Ablauf,Traces,Anteil,Median (Tage),Durchschnitt (Tage)
0,P10 > P36 > P5✅5 > T⬆️ > T⬇️ > D10 > D36 > D5✅5 > 🪙❓ > 🪙❗,1369,1369/6449 [21.23%],55.201076,65.890619
1,P10 > P36 > P31 > P5✅5 > T⬆️ > T⬇️ > D10 > D36 > D31 > D5✅5 > 🪙❓ > 🪙❗,624,624/6449 [9.68%],65.215041,73.488146
2,P10 > P5✅5 > T⬆️ > T⬇️ > D10 > D5✅5 > 🪙❓ > 🪙❗,361,361/6449 [5.60%],49.279769,60.630991
3,P10 > P36 > P5✅5 > T⬆️ > T⬇️ > D10 > D5❌6 > D5❌0 > D10 > D36 > D5✅5 > 🪙❓ > 🪙❗,311,311/6449 [4.82%],66.053854,72.777658
4,P10 > P32 > P5✅5 > T⬆️ > T⬇️ > D10 > D32 > D5✅5 > 🪙❓ > 🪙❗,254,254/6449 [3.94%],53.708142,63.440400
...,...,...,...,...,...
748,P10 > P36 > P35 > P5✅7 > T⬆️ > D10 > D5❌6 > T⬇️ > D5❌0 > D10 > D36 > D35 > D5✅7 > 🪙❓ > 🪙❗,1,1/6449 [0.02%],83.750255,83.750255
749,T⬆️ > T⬇️ > P10 > P36 > P35 > P5✅7 > D10 > D5❌6 > D5❌0 > D10 > D36 > D35 > D5✅7 > 🪙❓ > 🪙❗,1,1/6449 [0.02%],24.729826,24.729826
750,P10 > P36 > P31 > P35 > P5✅7 > T⬆️ > T⬇️ > D10 > D5❌6 > D5❌0 > D10 > D5❌6 > D5❌0 > D10 > D36 > D5✅5 > 🪙❓ > 🪙❗,1,1/6449 [0.02%],267.198241,267.198241
751,P10 > P36 > P5✅5 > T⬆️ > T⬇️ > D10 > D5❌6 > D5❌0 > D10 > D36 > D31 > D35 > D5✅7 > 🪙❓ > 🪙❗,1,1/6449 [0.02%],99.312488,99.312488


...

,Ablauf,Traces,Anteil,Median (Tage),Durchschnitt (Tage)
0,P10 > P36 > P5✅5 > D10 > D36 > D5✅5 > 🪙❓ > 🪙❗,1776,1776/4733 [37.52%],46.204144,60.051519
1,P10 > P36 > P31 > P5✅5 > D10 > D36 > D31 > D5✅5 > 🪙❓ > 🪙❗,877,877/4733 [18.53%],53.018403,63.007440
2,P10 > P36 > P5✅5 > D10 > D5❌6 > D5❌0 > D10 > D36 > D5✅5 > 🪙❓ > 🪙❗,374,374/4733 [7.90%],61.949433,69.605391
3,P10 > P36 > P31 > P5✅5 > D10 > D5❌6 > D5❌0 > D10 > D36 > D31 > D5✅5 > 🪙❓ > 🪙❗,243,243/4733 [5.13%],64.000451,71.458927
4,P10 > P36 > P5✅5 > 📨 > D10 > D36 > D5✅5 > 🪙❓ > 🪙❗,141,141/4733 [2.98%],81.036042,89.833088
...,...,...,...,...,...
215,P10 > P36 > P31 > P35 > P5✅7 > D10 > D5❌6 > D5❌0 > D10 > D36 > D5❌1 > D5❌0 > D10 > D36 > D31 > D5✅5 > 🪙❓ > 🪙❗,1,1/4733 [0.02%],136.199896,136.199896
216,P10 > P36 > P31 > P35 > P5✅7 > D10 > D5❌6 > D5❌0 > D10 > D5❌6 > D5❌0 > D10 > D5❌6 > D5❌0 > D10 > D36 > D31 > D35 > D5✅7 > 🪙❓ > 🪙❗,1,1/4733 [0.02%],91.938472,91.938472
217,P10 > P36 > P31 > P5❌5 > P5❌0 > P10 > P5❌6 > P5❌0 > P10 > P36 > P31 > P5✅5 > D10 > D36 > D31 > D5❌5 > D5❌0 > D10 > D36 > D31 > D5✅5 > 🪙❓ > 🪙❗,1,1/4733 [0.02%],57.180012,57.180012
218,P10 > P36 > P31 > P35 > P5✅7 > D10 > D5❌6 > D5❌0 > D10 > D5❌6 > D5❌0 > D10 > D36 > D5✅5 > 🪙❓ > 🪙❗,1,1/4733 [0.02%],267.198241,267.198241



int_without_trips_with_year_marker: 386 Abläufe


,Ablauf,Traces,Anteil,Median (Tage),Durchschnitt (Tage)
0,?? > P10 > P36 > P5✅5 > D10 > D36 > D5✅5 > 🪙❓ > 🪙❗,1776,1776/6449 [27.54%],206.646562,210.660984
1,?? > P10 > P36 > P31 > P5✅5 > D10 > D36 > D31 > D5✅5 > 🪙❓ > 🪙❗,877,877/6449 [13.60%],217.646597,218.192474
2,P10 > P5✅5 > D10 > D5✅5 > 🪙❓ > 🪙❗ > ??,407,407/6449 [6.31%],227.571065,216.088388
3,?? > P10 > P36 > P5✅5 > D10 > D5❌6 > D5❌0 > D10 > D36 > D5✅5 > 🪙❓ > 🪙❗,374,374/6449 [5.80%],236.646713,227.945621
4,P10 > P32 > P5✅5 > D10 > D32 > D5✅5 > 🪙❓ > 🪙❗ > ??,293,293/6449 [4.54%],208.526146,202.947565
...,...,...,...,...,...
381,P10 > ?? > P36 > P35 > P5✅7 > 📨 > D10 > D5❌6 > D5❌0 > D10 > D5❌6 > D5❌0 > D10 > D36 > D5✅5 > 🪙❓ > 🪙❗,1,1/6449 [0.02%],206.058611,206.058611
382,P10 > P32 > P5✅5 > D10 > D5❌2 > D5❌0 > D10 > D32 > D5✅5 > 🪙❓ > ?? > 🪙❗,1,1/6449 [0.02%],101.052674,101.052674
383,P10 > P5❌5 > ?? > P5❌0 > P10 > P36 > P5❌1 > P5❌0 > P10 > P36 > P5❌1 > P5❌0 > P10 > P36 > P31 > P5✅5 > 📨 > 📨 > D10 > D36 > D31 > D5✅5 > 🪙❓ > 🪙❗,1,1/6449 [0.02%],127.293843,127.293843
384,?? > P10 > P36 > P31 > P35 > P5✅7 > D10 > D5❌6 > D5❌0 > D10 > D5❌6 > D5❌0 > D10 > D36 > D5✅5 > 🪙❓ > 🪙❗,1,1/6449 [0.02%],276.646713,276.646713



dom_with_year_marker: 116 Abläufe


,Ablauf,Traces,Anteil,Median (Tage),Durchschnitt (Tage)
0,?? > D10 > D36 > D5✅5 > 🪙❓ > 🪙❗,4610,4610/10500 [43.90%],185.646910,195.518331
1,?? > D10 > D36 > D31 > D5✅5 > 🪙❓ > 🪙❗,2472,2472/10500 [23.54%],185.646910,197.744868
2,D10 > D5✅5 > 🪙❓ > 🪙❗ > ??,1348,1348/10500 [12.84%],180.456557,176.635063
3,D10 > D32 > D5✅5 > 🪙❓ > 🪙❗ > ??,563,563/10500 [5.36%],188.567755,170.734077
4,?? > D10 > D5❌6 > D5❌0 > D10 > D36 > D5✅5 > 🪙❓ > 🪙❗,344,344/10500 [3.28%],199.647245,215.948360
...,...,...,...,...,...
111,D10 > D5✅5 > D5❌_ > D10 > D5❌5 > D5❌0 > D10 > D5✅5 > 🪙❓ > 🪙❗ > ??,1,1/10500 [0.01%],342.085266,342.085266
112,D20 > 🪙❓ > 🪙❗ > ??,1,1/10500 [0.01%],233.441690,233.441690
113,D10 > D5❌2 > D5❌0 > D10 > D32 > D5✅5 > ?? > 🪙❓ > 🪙❗,1,1/10500 [0.01%],172.390556,172.390556
114,D10 > D42 > D5❌_ > ??,1,1/10500 [0.01%],65.529363,65.529363


In [18]:
start_end(logs)

### Trace-Varianten je Log: nur Declaration

Alle anderen Events werden vor der Variantenbildung entfernt. Die Ablaeufe verwenden die Kurzcodes aus der Legende oben; die Anteile beziehen sich auf Traces mit Declaration-Events.

In [19]:
declaration_variants = {}
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    for name, log in logs.items():
        declaration_events = log[log["concept:name"].str.contains(r"\bDeclaration\b", na=False)]
        declaration_variants[name] = variants(declaration_events)
        #print("=" * 70)
        #print(f"\n{name}: {len(declaration_variants[name])} Declaration-Ablaeufe")
        #display(declaration_variants[name])

In [20]:
import re
from collections import Counter
from python_analysis.functions import variants

declaration_events = logs["domestic"][
    logs["domestic"]["concept:name"].str.contains(
        r"\bDeclaration\b", na=False
    )
]

def split_variants(log, event: str):
    counts = Counter()
    for row in variants(log).itertuples(index=False):
        # D10 bleibt erhalten; ein D10 am Trace-Anfang erzeugt keinen leeren Teil.
        parts = re.split(r" > (?="+event+"(?: > |$))", row.Ablauf)
        for part in parts:
            counts[part] += row.Traces

    split_variants = (
        pd.DataFrame(counts.items(), columns=["Ablauf", "Teil-Traces"])
        .sort_values("Teil-Traces", ascending=False, ignore_index=True)
    )
    total = split_variants["Teil-Traces"].sum()
    split_variants["Anteil"] = split_variants["Teil-Traces"].map(
        lambda n: f"{n}/{total} [{n / total:.2%}]"
    )

    return split_variants

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    df = split_variants(declaration_events, "D10") # Declaration Submitted by EMPLOYEE
    display(df)

,Ablauf,Teil-Traces,Anteil
0,D10 > D36 > D5✅5,5133,5133/11666 [44.00%]
1,D10 > D36 > D31 > D5✅5,2788,2788/11666 [23.90%]
2,D10 > D5✅5,1462,1462/11666 [12.53%]
3,D10 > D5❌6 > D5❌0,928,928/11666 [7.95%]
4,D10 > D32 > D5✅5,659,659/11666 [5.65%]
5,D10 > D36 > D5❌5 > D5❌0,185,185/11666 [1.59%]
6,D20,135,135/11666 [1.16%]
7,D10 > D5❌2 > D5❌0,85,85/11666 [0.73%]
8,D10 > D5✅5 > D5❌_,74,74/11666 [0.63%]
9,D10 > D5❌5 > D5❌0,60,60/11666 [0.51%]


In [21]:
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    print("=" * 35 + " dom_log_base " + "=" * 35)
    df_variants = variants(logs["domestic"])
    df_split = split_variants(logs["domestic"], "D20") # Declaration Submitted by EMPLOYEE
    display(df_variants.loc[df_variants["Ablauf"].str.contains("D20", regex=False, na=False)])
    display(df_split.loc[df_split["Ablauf"].str.contains("D20", regex=False, na=False)])

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    print("=" * 35 + " int_log_base " + "=" * 35)
    df_variants = variants(logs["int_without_trips"])
    df_split = split_variants(logs["int_without_trips"], "D20") # Declaration Submitted by EMPLOYEE
    display(df_variants.loc[df_variants["Ablauf"].str.contains("D20", regex=False, na=False)])
    display(df_split.loc[df_split["Ablauf"].str.contains("D20", regex=False, na=False)])

=================================== dom_log_base ===================================


,Ablauf,Traces,Anteil,Median (Tage),Durchschnitt (Tage)
7,D20,134,134/10500 [1.28%],0.000000,0.000000
96,D20 > 🪙❓ > 🪙❗,1,1/10500 [0.01%],20.088113,20.088113


,Ablauf,Teil-Traces,Anteil
7,D20,134,134/10500 [1.28%]
96,D20 > 🪙❓ > 🪙❗,1,1/10500 [0.01%]


=================================== int_log_base ===================================


,Ablauf,Traces,Anteil,Median (Tage),Durchschnitt (Tage)
32,P10 > P36 > P5✅5 > D20,24,24/6449 [0.37%],46.631684,45.412224
49,P10 > P36 > P31 > P5✅5 > D20,11,11/6449 [0.17%],41.559630,59.455667
52,D20,10,10/6449 [0.16%],0.000000,0.000000
56,P10 > P36 > P5✅5 > 📨 > D20,9,9/6449 [0.14%],60.088356,75.457049
87,P10 > P5✅5 > D20,5,5/6449 [0.08%],25.812338,44.782604
94,P10 > P36 > P31 > P35 > P5✅7 > D20,4,4/6449 [0.06%],37.481707,37.222948
113,P10 > P36 > P31 > P5✅5 > 📨 > D20,3,3/6449 [0.05%],61.113889,65.276871
181,P10 > P35 > P5✅7 > D20,2,2/6449 [0.03%],18.943189,18.943189
185,P10 > P5✅5 > D20 > P5❌_,2,2/6449 [0.03%],102.834844,102.834844
220,P10 > P36 > P31 > P5✅5 > 📨 > D10 > D5❌6 > D5❌0 > D10 > D5❌6 > D5❌0 > D20,1,1/6449 [0.02%],93.954641,93.954641


,Ablauf,Teil-Traces,Anteil
12,D20,72,72/6514 [1.11%]
115,D20 > P5❌_,3,3/6514 [0.05%]


In [22]:
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    df_variants = variants(logs["domestic"])
    display(df_variants.loc[df_variants["Ablauf"].str.contains("🪙", regex=False, na=False)])

    df_variants = variants(logs["int_without_trips"])
    display(df_variants.loc[df_variants["Ablauf"].str.contains("🪙", regex=False, na=False)])

,Ablauf,Traces,Anteil,Median (Tage),Durchschnitt (Tage)
0,D10 > D36 > D5✅5 > 🪙❓ > 🪙❗,4618,4618/10500 [43.98%],7.036163,9.377247
1,D10 > D36 > D31 > D5✅5 > 🪙❓ > 🪙❗,2473,2473/10500 [23.55%],10.232593,12.550129
2,D10 > D5✅5 > 🪙❓ > 🪙❗,1392,1392/10500 [13.26%],6.311244,9.891926
3,D10 > D32 > D5✅5 > 🪙❓ > 🪙❗,575,575/10500 [5.48%],7.310475,9.966244
4,D10 > D5❌6 > D5❌0 > D10 > D36 > D5✅5 > 🪙❓ > 🪙❗,345,345/10500 [3.29%],11.104410,18.572796
5,D10 > D5❌6 > D5❌0 > D10 > D36 > D31 > D5✅5 > 🪙❓ > 🪙❗,188,188/10500 [1.79%],15.061852,23.858456
8,D10 > D36 > D5❌5 > D5❌0 > D10 > D36 > D5✅5 > 🪙❓ > 🪙❗,77,77/10500 [0.73%],13.290833,20.007310
9,D10 > D5❌2 > D5❌0 > D10 > D32 > D5✅5 > 🪙❓ > 🪙❗,57,57/10500 [0.54%],11.006597,17.590916
10,D10 > D36 > D5❌5 > D5❌0 > D10 > D36 > D31 > D5✅5 > 🪙❓ > 🪙❗,49,49/10500 [0.47%],15.790660,19.509727
11,D10 > D5✅5 > D5❌_ > D10 > D5✅5 > 🪙❓ > 🪙❗,40,40/10500 [0.38%],14.078148,22.539657


,Ablauf,Traces,Anteil,Median (Tage),Durchschnitt (Tage)
0,P10 > P36 > P5✅5 > D10 > D36 > D5✅5 > 🪙❓ > 🪙❗,1776,1776/6449 [27.54%],46.204144,60.051519
1,P10 > P36 > P31 > P5✅5 > D10 > D36 > D31 > D5✅5 > 🪙❓ > 🪙❗,877,877/6449 [13.60%],53.018403,63.007440
2,P10 > P5✅5 > D10 > D5✅5 > 🪙❓ > 🪙❗,415,415/6449 [6.44%],45.159745,57.584601
3,P10 > P36 > P5✅5 > D10 > D5❌6 > D5❌0 > D10 > D36 > D5✅5 > 🪙❓ > 🪙❗,374,374/6449 [5.80%],61.949433,69.605391
4,P10 > P32 > P5✅5 > D10 > D32 > D5✅5 > 🪙❓ > 🪙❗,304,304/6449 [4.71%],49.121881,62.414359
5,P10 > P36 > P31 > P5✅5 > D10 > D5❌6 > D5❌0 > D10 > D36 > D31 > D5✅5 > 🪙❓ > 🪙❗,243,243/6449 [3.77%],64.000451,71.458927
6,P10 > P36 > P5✅5 > 📨 > D10 > D36 > D5✅5 > 🪙❓ > 🪙❗,141,141/6449 [2.19%],81.036042,89.833088
7,D10 > D5✅5 > 🪙❓ > 🪙❗,111,111/6449 [1.72%],7.304248,11.417226
8,P10 > P36 > P35 > P5✅7 > D10 > D36 > D5✅5 > 🪙❓ > 🪙❗,109,109/6449 [1.69%],91.364769,99.606850
9,P10 > P36 > P31 > P35 > P5✅7 > D10 > D36 > D31 > D5✅5 > 🪙❓ > 🪙❗,91,91/6449 [1.41%],82.271146,96.219958


In [23]:
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    df_variants = variants(declaration_events)
    df_split = split_variants(declaration_events, "D5❌") # Declaration Submitted by EMPLOYEE
    display(df_variants.loc[df_variants["Ablauf"].str.contains("D5❌", regex=False, na=False)])
    display(df_split.loc[df_split["Ablauf"].str.contains("D5❌", regex=False, na=False)])

,Ablauf,Traces,Anteil,Median (Tage),Durchschnitt (Tage)
4,D10 > D5❌6 > D5❌0 > D10 > D36 > D5✅5,345,345/10500 [3.29%],5.750509,12.156122
5,D10 > D5❌6 > D5❌0 > D10 > D36 > D31 > D5✅5,188,188/10500 [1.79%],7.846615,17.185605
6,D10 > D5❌6 > D5❌0,174,174/10500 [1.66%],0.896800,8.415967
8,D10 > D36 > D5❌5 > D5❌0 > D10 > D36 > D5✅5,77,77/10500 [0.73%],6.949479,13.736680
9,D10 > D5❌2 > D5❌0 > D10 > D32 > D5✅5,57,57/10500 [0.54%],3.586481,6.598261
10,D10 > D36 > D5❌5 > D5❌0 > D10 > D36 > D31 > D5✅5,49,49/10500 [0.47%],10.724641,14.485624
11,D10 > D5✅5 > D5❌_ > D10 > D5✅5,40,40/10500 [0.38%],7.113987,15.835213
12,D10 > D5❌6 > D5❌0 > D10 > D5❌6 > D5❌0 > D10 > D36 > D5✅5,36,36/10500 [0.34%],6.630764,23.878389
13,D10 > D36 > D5❌5 > D5❌0,31,31/10500 [0.30%],3.970845,8.258822
14,D10 > D5❌6 > D5❌0 > D10 > D5❌6 > D5❌0 > D10 > D36 > D31 > D5✅5,29,29/10500 [0.28%],13.120602,24.490311


,Ablauf,Teil-Traces,Anteil
4,D10 > D5❌6 > D5❌0 > D10 > D36 > D5✅5,345,345/10500 [3.29%]
5,D10 > D5❌6 > D5❌0 > D10 > D36 > D31 > D5✅5,188,188/10500 [1.79%]
6,D10 > D5❌6 > D5❌0,174,174/10500 [1.66%]
8,D10 > D36 > D5❌5 > D5❌0 > D10 > D36 > D5✅5,77,77/10500 [0.73%]
9,D10 > D5❌2 > D5❌0 > D10 > D32 > D5✅5,57,57/10500 [0.54%]
10,D10 > D36 > D5❌5 > D5❌0 > D10 > D36 > D31 > D5✅5,49,49/10500 [0.47%]
11,D10 > D5✅5 > D5❌_ > D10 > D5✅5,40,40/10500 [0.38%]
12,D10 > D5❌6 > D5❌0 > D10 > D5❌6 > D5❌0 > D10 > D36 > D5✅5,36,36/10500 [0.34%]
13,D10 > D36 > D5❌5 > D5❌0,31,31/10500 [0.30%]
14,D10 > D5❌6 > D5❌0 > D10 > D5❌6 > D5❌0 > D10 > D36 > D31 > D5✅5,29,29/10500 [0.28%]


In [24]:
dom_base_neighbors = direct_neighborhood(logs["domestic"])
int_base_neighbors = direct_neighborhood(logs["int_without_trips"])
display(dom_base_neighbors)
display(int_base_neighbors)

,Vorbereich,Nachbereich
Code,,
D10,"{D5❌_, D5❌5, D5❌2, D46, D5❌6, D5❌0, D5❌1}","{D32, D42, D5❌5, D5❌2, D46, D5❌6, D5❌0, D45, D..."
D20,{},{🪙❓}
D31,{D36},"{D5❌5, D5✅5}"
D32,{D10},"{D5❌5, D5✅5}"
D36,{D10},"{D31, D5❌5, D5❌1, D5✅5}"
D42,{D10},{D5❌_}
D45,{D10},{D5❌_}
D46,{D10},{D10}
D5✅5,"{D31, D10, D32, D36}","{D5❌_, 🪙❓, 🪙❗}"


,Vorbereich,Nachbereich
Code,,
D10,"{P5✅5, D5❌_, 📨, P35, D5❌5, P5✅7, D5❌6, D5❌0, P32}","{D5✅7, D32, P5❌_, D5❌5, D5❌2, D5✅5, D5❌6, D5❌0..."
D20,"{P5✅5, D5❌0, 📨, P5✅7}",{P5❌_}
D31,{D36},"{D35, D5❌5, D5✅5}"
D32,"{P5✅5, D10}","{D5❌5, D5✅7, D35, D5✅5}"
D35,"{D5✅7, D32, D31, D10, D36}","{D5✅7, D5❌7, 🪙❓}"
D36,{D10},"{P5❌_, D5❌5, D5✅5, D31, D5❌1, D35}"
D5✅5,"{D32, P5❌_, D31, D10, D36}","{D5❌_, 🪙❓, P5❌_, 🪙❗}"
D5✅7,"{D10, D32, D35}","{D5❌_, 🪙❓, D35}"
D5❌0,"{D5❌7, P5❌_, D5❌5, D5❌2, D5❌6, D10, D5❌1}","{P5❌_, D10, D20}"


In [25]:
neighbors_dom_decl = direct_neighborhood(logs["domestic"].loc[logs["domestic"]["concept:name"].str.contains("Declaration", na=False)])
neighbors_int_decl = direct_neighborhood(logs["international"].loc[logs["international"]["concept:name"].str.contains("Declaration", na=False)])

In [26]:
display(neighbors_dom_decl.loc[neighbors_dom_decl.index.str.contains("D5❌")])
display(neighbors_dom_decl.loc[neighbors_dom_decl.index.str.contains("D10")])
print("*" *70)
display(neighbors_int_decl.loc[neighbors_int_decl.index.str.contains("D5❌")])
display(neighbors_int_decl.loc[neighbors_int_decl.index.str.contains("D10")])

,Vorbereich,Nachbereich
Code,,
D5❌0,"{D5❌5, D5❌2, D5❌6, D10, D5❌1}",{D10}
D5❌1,{D36},"{D10, D5❌0}"
D5❌2,{D10},"{D10, D5❌0}"
D5❌5,"{D31, D10, D32, D36}","{D10, D5❌0}"
D5❌6,{D10},"{D10, D5❌0}"
D5❌_,"{D42, D45, D5✅5}",{D10}


,Vorbereich,Nachbereich
Code,,
D10,"{D5❌_, D5❌5, D5❌2, D46, D5❌6, D5❌0, D5❌1}","{D32, D42, D5❌5, D5❌2, D46, D5❌6, D5❌0, D45, D..."


**********************************************************************


,Vorbereich,Nachbereich
Code,,
D5❌0,"{D5❌7, D5❌5, D5❌2, D5❌6, D10, D5❌1}","{D10, D20}"
D5❌1,{D36},{D5❌0}
D5❌2,{D10},{D5❌0}
D5❌5,"{D31, D10, D32, D36}","{D10, D5❌0}"
D5❌6,{D10},"{D10, D5❌0}"
D5❌7,{D35},{D5❌0}
D5❌_,"{D5✅7, D5✅5}",{D10}


,Vorbereich,Nachbereich
Code,,
D10,"{D5❌6, D5❌0, D5❌_, D5❌5}","{D5✅7, D32, D5❌5, D5❌2, D5✅5, D5❌6, D5❌0, D36,..."


In [27]:
log = logs["domestic"]
declarations = log.loc[
    log["concept:name"].str.startswith("Declaration ", na=False)
]
display(predecessor_times(declarations, target="D5❌0").style.format({
    "Min_Tage": "{:.3f}",
    "Max_Tage": "{:.3f}",
    "Median_Tage": "{:.3f}",
    "Durchschnitt_Tage": "{:.3f}",
}))
display(successor_times(declarations, target="D5❌0").style.format({
    "Min_Tage": "{:.3f}",
    "Max_Tage": "{:.3f}",
    "Median_Tage": "{:.3f}",
    "Durchschnitt_Tage": "{:.3f}",
}))

,Anzahl,Traces,Min_Tage,Max_Tage,Median_Tage,Durchschnitt_Tage
Voraktivitaet,,,,,,
D5❌6,928,837,0.001,27.163,0.922,2.044
D5❌5,287,275,0.003,55.188,1.051,3.711
D5❌2,85,80,0.001,6.133,0.916,1.437
D5❌1,58,57,0.033,6.370,1.023,2.135
D10,7,7,0.000,0.000,0.000,0.000


,Anzahl,Traces,Min_Tage,Max_Tage,Median_Tage,Durchschnitt_Tage
Nachaktivitaet,,,,,,
D10,1081,961,0.000,356.922,0.202,5.179


In [28]:
from python_analysis.functions import rejection_triples

display(rejection_triples(logs["domestic"]))

Triple,Anzahl,Traces,Prozent
D5❌6 → D5❌0 → D10,732,652,53.63 %
D5❌5 → D5❌0 → D10,221,216,16.19 %
D5❌6 → D5❌0 → <Trace-Ende>,196,196,14.36 %
D5❌2 → D5❌0 → D10,77,72,5.64 %
D5❌5 → D5❌0 → <Trace-Ende>,65,65,4.76 %
D5❌1 → D5❌0 → D10,45,44,3.30 %
D5❌1 → D5❌0 → <Trace-Ende>,13,13,0.95 %
D5❌2 → D5❌0 → <Trace-Ende>,8,8,0.59 %
D10 → D5❌0 → D10,6,6,0.44 %
D10 → D5❌0 → <Trace-Ende>,1,1,0.07 %


Domestic: \
D5x0 | kommt von D5x* in 1349 von 1357 Fällen (99,41%) | geht auf Trace-Ende oder D10 \
D5x1 | kommt von D36 (100%) |
D5x2 | kommt von D10 (100%) |
D5x5 | kommt nach D10 () oder D3* () |
D5x6 | Kommt von D10 (100%)
D5x_ |

In [29]:
display(rejection_triples(logs["int_without_trips"]))

Triple,Anzahl,Traces,Prozent
D5❌6 → D5❌0 → D10,1386,1147,77.87 %
D5❌6 → D5❌0 → <Trace-Ende>,138,138,7.75 %
D5❌5 → D5❌0 → D10,108,105,6.07 %
D5❌2 → D5❌0 → D10,74,72,4.16 %
D5❌1 → D5❌0 → D10,34,34,1.91 %
D5❌5 → D5❌0 → <Trace-Ende>,14,14,0.79 %
D5❌2 → D5❌0 → <Trace-Ende>,7,7,0.39 %
D5❌1 → D5❌0 → <Trace-Ende>,6,6,0.34 %
D10 → D5❌0 → D10,5,5,0.28 %
D5❌7 → D5❌0 → D10,3,3,0.17 %


In [30]:
display(rejection_triples(logs["int_decl_only_without_trips"]))

Triple,Anzahl,Traces,Prozent
D5❌6 → D5❌0 → D10,795,707,54.56 %
D5❌5 → D5❌0 → D10,231,225,15.85 %
D5❌6 → D5❌0 → <Trace-Ende>,206,206,14.14 %
D5❌2 → D5❌0 → D10,84,79,5.77 %
D5❌5 → D5❌0 → <Trace-Ende>,66,66,4.53 %
D5❌1 → D5❌0 → D10,46,45,3.16 %
D5❌1 → D5❌0 → <Trace-Ende>,13,13,0.89 %
D5❌2 → D5❌0 → <Trace-Ende>,8,8,0.55 %
D10 → D5❌0 → D10,6,6,0.41 %
D10 → D5❌0 → <Trace-Ende>,1,1,0.07 %


In [31]:
display(rejection_triples(logs["int_permit_without_trips"]))

Triple,Anzahl,Traces,Prozent
D5❌6 → D5❌0 → D10,1323,1092,78.38 %
D5❌6 → D5❌0 → <Trace-Ende>,128,128,7.58 %
D5❌5 → D5❌0 → D10,98,96,5.81 %
D5❌2 → D5❌0 → D10,67,65,3.97 %
D5❌1 → D5❌0 → D10,33,33,1.95 %
D5❌5 → D5❌0 → <Trace-Ende>,13,13,0.77 %
D5❌2 → D5❌0 → <Trace-Ende>,7,7,0.41 %
D5❌1 → D5❌0 → <Trace-Ende>,6,6,0.36 %
D10 → D5❌0 → D10,5,5,0.30 %
D5❌7 → D5❌0 → D10,3,3,0.18 %


In [32]:
from python_analysis.functions import bottleneck_analysis, event_code

log = logs["domestic"]
intervals = bottleneck_analysis(log)["intervals"].copy()
intervals["Von"] = intervals["from_activity"].map(event_code)
intervals["Nach"] = intervals["to_activity"].map(event_code)

stats = (
    intervals.groupby(["Von", "Nach"])
    .agg(
        Anzahl=("elapsed_days", "size"),
        Min_Tage=("elapsed_days", "min"),
        Max_Tage=("elapsed_days", "max"),
    )
    .reset_index()
)

zero = stats.loc[
    stats["Min_Tage"].eq(0) & stats["Max_Tage"].eq(0)
]

for activity in sorted(log["concept:name"].map(event_code).unique()):
    before = zero.loc[zero["Nach"].eq(activity)]
    after = zero.loc[zero["Von"].eq(activity)]

    if before.empty and after.empty:
        continue

    print(f"\n=== {activity} ===")

    if not before.empty:
        print("Vorbereich:")
        display(
            before.drop(columns="Nach")
            .set_index("Von")
            .rename_axis("Voraktivitaet")
        )

    if not after.empty:
        print("Nachbereich:")
        display(
            after.drop(columns="Von")
            .set_index("Nach")
            .rename_axis("Nachaktivitaet")
        )

In [33]:
from python_analysis.functions import bottleneck_analysis, event_code

log = logs["international"]
intervals = bottleneck_analysis(log)["intervals"].copy()
intervals["Von"] = intervals["from_activity"].map(event_code)
intervals["Nach"] = intervals["to_activity"].map(event_code)

stats = (
    intervals.groupby(["Von", "Nach"])
    .agg(
        Anzahl=("elapsed_days", "size"),
        Min_Tage=("elapsed_days", "min"),
        Max_Tage=("elapsed_days", "max"),
    )
    .reset_index()
)

zero = stats.loc[
    stats["Min_Tage"].eq(0) & stats["Max_Tage"].eq(0)
]

for activity in sorted(log["concept:name"].map(event_code).unique()):
    before = zero.loc[zero["Nach"].eq(activity)]
    after = zero.loc[zero["Von"].eq(activity)]

    if before.empty and after.empty:
        continue

    print(f"\n=== {activity} ===")

    if not before.empty:
        print("Vorbereich:")
        display(
            before.drop(columns="Nach")
            .set_index("Von")
            .rename_axis("Voraktivitaet")
        )

    if not after.empty:
        print("Nachbereich:")
        display(
            after.drop(columns="Von")
            .set_index("Nach")
            .rename_axis("Nachaktivitaet")
        )


=== D35 ===
Vorbereich:


,Anzahl,Min_Tage,Max_Tage
Voraktivitaet,,,
D5✅7,3,0.0,0.0



=== D5✅7 ===
Nachbereich:


,Anzahl,Min_Tage,Max_Tage
Nachaktivitaet,,,
D35,3,0.0,0.0



=== P32 ===
Vorbereich:


...

,Anzahl,Min_Tage,Max_Tage
Voraktivitaet,,,
P5✅7,3,0.0,0.0



=== P5✅5 ===
Nachbereich:


,Anzahl,Min_Tage,Max_Tage
Nachaktivitaet,,,
P32,1,0.0,0.0



=== P5✅7 ===
Nachbereich:


,Anzahl,Min_Tage,Max_Tage
Nachaktivitaet,,,
P35,3,0.0,0.0


mögliche Kopplung zeigen deine drei Paare:

 Beobachteter Übergang | Mögliche Interpretation

 D5✅7 → D35 | Finale Declaration-Freigabe durch Direktor setzt Supervisor-Freigabe mit \
 P5✅7 → P35 | Finale Permit-Freigabe durch Direktor setzt Supervisor-Freigabe mit \
 P5✅5 → P32 | Finale Permit-Freigabe durch Supervisor setzt Pre-Approver-Freigabe mit

könnte auch eine einzige Systemaktion mit zwei Logeinträgen sein, statt zweier tatsächlich separat ausgeführter Arbeitsschritte.

In [34]:
log = logs["domestic"]
declarations = log.loc[
    log["concept:name"].str.startswith("Declaration ", na=False)
]
display(predecessor_times(declarations, target="D5❌_").style.format({
    "Min_Tage": "{:.3f}",
    "Max_Tage": "{:.3f}",
    "Median_Tage": "{:.3f}",
    "Durchschnitt_Tage": "{:.3f}",
}))
display(successor_times(declarations, target="D5❌_").style.format({
    "Min_Tage": "{:.3f}",
    "Max_Tage": "{:.3f}",
    "Median_Tage": "{:.3f}",
    "Durchschnitt_Tage": "{:.3f}",
}))
log = logs["international"]
declarations = log.loc[
    log["concept:name"].str.startswith("Declaration ", na=False)
]
display(predecessor_times(declarations, target="D5❌_").style.format({
    "Min_Tage": "{:.3f}",
    "Max_Tage": "{:.3f}",
    "Median_Tage": "{:.3f}",
    "Durchschnitt_Tage": "{:.3f}",
}))
display(successor_times(declarations, target="D5❌_").style.format({
    "Min_Tage": "{:.3f}",
    "Max_Tage": "{:.3f}",
    "Median_Tage": "{:.3f}",
    "Durchschnitt_Tage": "{:.3f}",
}))

,Anzahl,Traces,Min_Tage,Max_Tage,Median_Tage,Durchschnitt_Tage
Voraktivitaet,,,,,,
D5✅5,89,85,0.013,293.360,3.107,11.398
D42,1,1,4.132,4.132,4.132,4.132
D45,1,1,0.794,0.794,0.794,0.794


,Anzahl,Traces,Min_Tage,Max_Tage,Median_Tage,Durchschnitt_Tage
Nachaktivitaet,,,,,,
D10,61,58,0.002,137.049,0.416,8.560


,Anzahl,Traces,Min_Tage,Max_Tage,Median_Tage,Durchschnitt_Tage
Voraktivitaet,,,,,,
D5✅5,94,89,0.073,45.055,2.793,4.743
D5✅7,9,9,0.105,26.874,2.189,5.705


,Anzahl,Traces,Min_Tage,Max_Tage,Median_Tage,Durchschnitt_Tage
Nachaktivitaet,,,,,,
D10,87,82,0.002,156.141,0.833,7.401


In [35]:
log = int_log_base  # alternativ: dom_log_base
case_id = "case:concept:name"
activities = ["Request Payment", "Payment Handled", "Send Reminder"]

counts = (
    log.loc[log["concept:name"].isin(activities)]
    .groupby([case_id, "concept:name"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=activities, fill_value=0)
)

repeated = counts.loc[counts.gt(1).any(axis=1)]

# Vollständige Traces behalten, nicht nur die drei gesuchten Aktivitäten.
matching_traces = (
    log.loc[log[case_id].isin(repeated.index)]
    .sort_values([case_id, "time:timestamp"], kind="stable")
)

print(f"{len(repeated)} von {log[case_id].nunique()} Traces betroffen")

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(repeated)                  # Häufigkeit jeder Aktivität je Fall
    display(variants(matching_traces)) # Vollständige Abläufe als Kurzcodes + Häufigkeit

28 von 6449 Traces betroffen


concept:name,Request Payment,Payment Handled,Send Reminder
case:concept:name,,,
declaration 1126,1,1,2
declaration 13321,1,1,2
declaration 1544,1,1,2
declaration 1886,1,1,2
declaration 27582,1,1,2
declaration 31495,1,1,2
declaration 31697,1,1,2
declaration 43745,1,1,2
declaration 44042,1,1,2


,Ablauf,Traces,Anteil,Median (Tage),Durchschnitt (Tage)
0,T⬆️ > T⬇️ > P10 > P36 > P5✅5 > 📨 > 📨 > D10 > D36 > D5✅5 > 🪙❓ > 🪙❗,5,5/28 [17.86%],359.771655,310.354938
1,P10 > P36 > P5✅5 > T⬆️ > T⬇️ > 📨 > 📨 > D10 > D36 > D5✅5 > 🪙❓ > 🪙❗,4,4/28 [14.29%],225.530943,218.980139
2,P10 > P36 > P31 > P5✅5 > T⬆️ > T⬇️ > 📨 > 📨 > D10 > D36 > D31 > D5✅5 > 🪙❓ > 🪙❗,3,3/28 [10.71%],217.167836,215.113032
3,P10 > P5✅5 > T⬆️ > T⬇️ > 📨 > 📨 > D10 > D36 > D5✅5 > 🪙❓ > 🪙❗,2,2/28 [7.14%],179.442182,179.442182
4,P10 > P36 > P5✅5 > D10 > T⬆️ > T⬇️ > D36 > D31 > D5✅5 > 🪙❓ > 🪙❗ > 📨 > 📨,1,1/28 [3.57%],153.755208,153.755208
5,T⬆️ > P10 > P36 > P31 > P5✅5 > T⬇️ > 📨 > 📨 > D10 > D36 > D31 > D5✅5 > 🪙❓ > 🪙❗,1,1/28 [3.57%],147.688519,147.688519
6,P10 > P36 > P31 > P5✅5 > T⬆️ > T⬇️ > 📨 > 📨 > D10 > D5❌6 > D5❌0 > D10 > D5❌6 > D5❌0 > D10 > D36 > D31 > D5✅5 > 🪙❓ > 🪙❗,1,1/28 [3.57%],223.049907,223.049907
7,P10 > P36 > P5✅5 > T⬆️ > T⬇️ > 📨 > 📨 > D10 > D5❌6 > D5❌0 > D10 > D36 > D5✅5 > 🪙❓ > 🪙❗,1,1/28 [3.57%],209.213634,209.213634
8,T⬆️ > P10 > P36 > P5✅5 > T⬇️ > 📨 > 📨 > D10 > D5❌6 > D5❌0 > D10 > D36 > D5✅5 > 🪙❓ > 🪙❗,1,1/28 [3.57%],175.771863,175.771863
9,P10 > P36 > P5✅5 > T⬆️ > T⬇️ > D10 > D5❌6 > D5❌0 > D10 > D36 > D5✅5 > 🪙❓ > 🪙❗ > 📨 > 📨,1,1/28 [3.57%],127.222465,127.222465


Trip:
Startet trip -> End trip
Keine Abhängigkeiten, parallel zum Rest

Payment:

Aufgabe 2

Aufgabe 3

Aufgabe 4

Aufgabe 5

Aufgabe 6

Aufgabe 7

Aufgabe 8

Aufgabe 9

Aufgabe 10

Aufgabe 11

In [36]:
from python_analysis.functions import rejection_timeline

analyses = {}

for name, log, entity in [
    ("Domestic Declaration", logs["domestic"], "Declaration"),
    ("International Declaration", logs["int_decl_only_without_trips"], "Declaration"),
    ("International Permit", logs["int_permit_without_trips"], "Permit")]:
    result = rejection_timeline(log, entity=entity)
    analyses[name] = result

    print(f"\n=== {name} ===")
    with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
        display(result["summary"].round(3))


=== Domestic Declaration ===


Anzahl  Traces  Median_vor_Tage  \
Monat   Voraktivitaet Nachaktivitaet                                    
2017-01 D5❌2          D10                  4       4            1.364   
        D5❌5          D10                  2       2            1.464   
2017-02 D10           D10                  1       1            0.000   
        D5❌2          D10                  4       4            1.195   
        D5❌5          <Trace-Ende>         2       2            0.537   
                      D10                  4       4            4.801   
2017-03 D5❌2          D10                  9       8            0.063   
        D5❌5          D10                  3       3            3.616   
2017-04 D5❌2          D10                  6       6            1.156   
        D5❌5          <Trace-Ende>         2       2            5.252   
                      D10                  4       4            0.577   
2017-05 D5❌2          <Trace-Ende>         1       1            2.678   
                      D10                 12      11            1.420   
        D5❌5          <Trace-Ende>         2       2            0.109   
                      D10                  6       6            3.137   
2017-06 D5❌2          <Trace-Ende>         1       1            0.242   
                      D10                  8       8            2.354   
        D5❌5          D10                  1       1            0.010   
2017-07 D5❌2          D10                  7       7            1.288   
        D5❌5          <Trace-Ende>         1       1            5.310   
                      D10                  2       2            0.150   
2017-08 D5❌5          D10                  1       1            4.100   
2017-09 D5❌2          D10                  2       2            0.547   
        D5❌5          <Trace-Ende>         4       4            1.032   
                      D10                  5       4            1.957   
2017-10 D5❌2          <Trace-Ende>         4       4            0.903   
                      D10                 12      10            1.065   
        D5❌5          D10                  4       4            0.668   
2017-11 D5❌2          D10                  7       7            0.255   
        D5❌5          <Trace-Ende>         1       1            0.050   
                      D10                  3       3            1.583   
2017-12 D5❌2          <Trace-Ende>         2       2            0.103   
                      D10                  6       6            1.494   
        D5❌5          <Trace-Ende>        10      10           11.787   
                      D10                  8       8            0.752   
        🪙❗            <Trace-Ende>         1       1            1.625   
2018-01 D5❌1          <Trace-Ende>         6       6            0.673   
                      D10                  1       1            0.214   
        D5❌5          <Trace-Ende>         6       6           28.955   
                      D10                  7       6            0.424   
        D5❌6          <Trace-Ende>         9       9            1.689   
                      D10                 33      32            1.190   
2018-02 D5❌1          <Trace-Ende>         1       1            1.884   
                      D10                 12      11            4.923   
        D5❌5          <Trace-Ende>         3       3            5.186   
                      D10                 22      21            1.116   
        D5❌6          <Trace-Ende>        12      12            0.431   
                      D10                 34      33            0.904   
2018-03 D5❌1          D10                  3       3            2.114   
        D5❌5          <Trace-Ende>         7       7            0.698   
                      D10                 18      18            0.951   
        D5❌6          <Trace-Ende>        12      12            0.655   
                      D10                 37      32            0.878   
2018-04 D10           D


=== International Declaration ===


Anzahl  Traces  Median_vor_Tage  \
Monat   Voraktivitaet Nachaktivitaet                                    
2017-01 D5❌2          D10                  5       5            1.811   
        D5❌5          D10                  4       3            1.914   
2017-02 D10           D10                  1       1            0.000   
        D5❌2          D10                  6       6            0.637   
        D5❌5          <Trace-Ende>         2       2            0.537   
                      D10                  8       8            5.194   
2017-03 D5❌2          D10                 10       9            0.090   
        D5❌5          D10                  3       3            3.616   
2017-04 D5❌2          D10                  8       8            1.582   
        D5❌5          <Trace-Ende>         2       2            5.252   
                      D10                  4       4            0.577   
2017-05 D5❌2          <Trace-Ende>         1       1            2.678   
                      D10                 13      12            0.795   
        D5❌5          <Trace-Ende>         2       2            0.109   
                      D10                  6       6            3.137   
2017-06 D5❌2          <Trace-Ende>         1       1            0.242   
                      D10                  8       8            2.354   
        D5❌5          D10                  2       2            0.154   
2017-07 D5❌2          D10                  7       7            1.288   
        D5❌5          <Trace-Ende>         1       1            5.310   
                      D10                  2       2            0.150   
2017-08 D5❌5          D10                  1       1            4.100   
2017-09 D5❌2          D10                  2       2            0.547   
        D5❌5          <Trace-Ende>         4       4            1.032   
                      D10                  5       4            1.957   
2017-10 D5❌2          <Trace-Ende>         4       4            0.903   
                      D10                 12      10            1.065   
        D5❌5          D10                  4       4            0.668   
2017-11 D5❌2          D10                  7       7            0.255   
        D5❌5          <Trace-Ende>         1       1            0.050   
                      D10                  3       3            1.583   
2017-12 D5❌2          <Trace-Ende>         2       2            0.103   
                      D10                  6       6            1.494   
        D5❌5          <Trace-Ende>        10      10           11.787   
                      D10                  8       8            0.752   
        🪙❗            <Trace-Ende>         1       1            1.625   
2018-01 D5❌1          <Trace-Ende>         6       6            0.673   
                      D10                  1       1            0.214   
        D5❌5          <Trace-Ende>         6       6           28.955   
                      D10                  9       8            0.503   
        D5❌6          <Trace-Ende>        11      11            2.930   
                      D10                 38      37            1.144   
2018-02 D5❌1          <Trace-Ende>         1       1            1.884   
                      D10                 12      11            4.923   
        D5❌5          <Trace-Ende>         4       4            5.154   
                      D10                 22      21            1.116   
        D5❌6          <Trace-Ende>        13      13            0.720   
                      D10                 57      54            1.170   
2018-03 D5❌1          D10                  3       3            2.114   
        D5❌5          <Trace-Ende>         7       7            0.698   
                      D10                 18      18            0.951   
        D5❌6          <Trace-Ende>        15      15            1.150   
                      D10                 51      43            1.415   
2018-04 D10           D


=== International Permit ===


Anzahl  Traces  Median_vor_Tage  \
Monat   Voraktivitaet Nachaktivitaet                                    
2017-01 P5❌5          P10                  1       1            0.065   
2017-03 P5❌2          P10                  3       3            2.834   
        P5❌5          P10                  3       3            0.638   
2017-04 P5❌2          P10                  8       7            0.040   
        P5❌5          P10                  1       1            4.148   
2017-05 P5❌2          P10                  4       4            0.752   
        P5❌5          P10                  7       7            2.233   
2017-06 P5❌2          P10                  3       3            0.040   
        P5❌5          P10                  1       1            0.597   
2017-07 P5❌5          P10                  2       2            2.112   
2017-08 P5❌2          P10                  1       1            4.847   
        P5❌5          P10                  1       1            0.025   
2017-09 P5❌2          P10                  5       5            0.826   
2017-10 P5❌5          P10                  3       3            7.974   
2017-11 P5❌2          P10                  1       1            5.070   
2018-01 P5❌1          P10                  4       3            0.828   
        P5❌5          P10                  3       3           29.787   
2018-02 P5❌1          P10                  2       2            3.982   
        P5❌5          P10                  1       1            1.230   
        P5❌6          P10                  6       5            1.105   
2018-03 P5❌1          P10                  5       5            1.801   
        P5❌5          P10                  4       4            1.013   
        P5❌6          P10                  4       4            1.533   
2018-04 P5❌1          P10                  2       2            0.922   
        P5❌5          P10                 10      10            4.121   
        P5❌6          P10                  2       2            0.787   
2018-05 P5❌5          P10                  7       7            0.970   
        P5❌6          P10                 11      11            0.548   
2018-06 P5❌1          P10                  6       6            1.361   
        P5❌5          P10                  9       9            5.105   
        P5❌6          P10                 16      16            0.727   
2018-07 P5❌1          P10                  5       5            1.105   
        P5❌5          P10                 13      13            5.000   
        P5❌6          P10                 10      10            1.196   
2018-08 P10           P10                  2       2            0.015   
        P5❌1          P10                  2       2            1.630   
        P5❌5          P10                  4       4            0.598   
        P5❌6          P10                 12      11            0.539   
2018-09 P5❌5          P10                  7       7            0.954   
        P5❌6          P10                  8       8            0.792   
        P5❌7          P10                  1       1            5.014   
2018-10 P5❌1          P10                  2       2            5.112   
        P5❌5          P10                  8       8            0.735   
        P5❌6          P10                  6       6            1.486   
2018-11 P5❌1          P10                  3       3            2.908   
        P5❌5          P10                  5       5            0.734   
        P5❌6          P10                  4       4            0.769   
2018-12 P5❌5          P10                  1       1            1.126   
        P5❌6          P10                  2       2            0.525   

                                      Median_nach_Tage  Null_vor_Anteil  \
Monat   Voraktivitaet Nachaktivitaet                                      
2017-01 P5❌5          P10                        0.007              0.0   
2017-03 P5❌2          P10                        0.042              0.0   
        P5❌5  

In [37]:
from python_analysis.functions import rejections_without_employee_followup
from python_analysis.functions import variants

missing_employee_followup = {}
for name in ("domestic", "international"):
    result = rejections_without_employee_followup(logs[name])
    missing_employee_followup[name] = result
    print(f"{name}: {len(result['cases'])} Traces mit insgesamt {len(result['events'])} Treffern")
    matching_traces = logs[name].loc[
        logs[name]["case:concept:name"].isin(result["cases"].index)
        & ~logs[name]["concept:name"].isin({"Start trip", "End trip"})
    ]
    with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
        display(variants(matching_traces))


domestic: 115 Traces mit insgesamt 124 Treffern


,Ablauf,Traces,Anteil,Median (Tage),Durchschnitt (Tage)
0,D10 > D5✅5 > D5❌_ > D10 > D5✅5 > 🪙❓ > 🪙❗,40,40/115 [34.78%],14.078148,22.539657
1,D10 > D5✅5 > D5❌_,21,21/115 [18.26%],0.807546,10.098172
2,D10 > D32 > D5✅5 > D5❌_ > D10 > D32 > D5✅5 > 🪙❓ > 🪙❗,7,7/115 [6.09%],24.062037,56.966915
3,D10 > D5❌6,5,5/115 [4.35%],114.757766,159.781819
4,D10 > D5❌6 > D10 > D5❌6 > D5❌0 > D10 > D36 > D5✅5 > 🪙❓ > 🪙❗,3,3/115 [2.61%],13.314201,81.505027
5,D10 > D5✅5 > 🪙❓ > D5❌_,3,3/115 [2.61%],29.876088,47.608318
6,D10 > D5❌6 > D10 > D5❌0 > D10 > D36 > D5✅5 > 🪙❓ > 🪙❗,2,2/115 [1.74%],7.152188,7.152188
7,D10 > D5❌6 > D10 > D5❌6 > D5❌0,2,2/115 [1.74%],2.609421,2.609421
8,D10 > D36 > D5❌5,2,2/115 [1.74%],9.421424,9.421424
9,D10 > D5❌2 > D5❌0 > D10 > D32 > D5✅5 > D5❌_ > D10 > D32 > D5✅5 > 🪙❓ > 🪙❗,2,2/115 [1.74%],15.131047,15.131047


international: 166 Traces mit insgesamt 178 Treffern


,Ablauf,Traces,Anteil,Median (Tage),Durchschnitt (Tage)
0,P10 > P5✅5 > D10 > D5✅5 > D5❌_ > D10 > D5✅5 > 🪙❓ > 🪙❗,25,25/166 [15.06%],62.365602,74.538307
1,P10 > P32 > P5✅5 > D10 > D32 > D5✅5 > D5❌_ > D10 > D32 > D5✅5 > 🪙❓ > 🪙❗,23,23/166 [13.86%],121.253727,119.723671
2,P10 > P32 > P5✅5 > D10 > D32 > D5✅5 > D5❌_,9,9/166 [5.42%],79.934699,67.688011
3,P10 > P5✅5 > P5❌_ > P10 > P5✅5 > D10 > D5✅5 > 🪙❓ > 🪙❗,6,6/166 [3.61%],71.330087,75.342544
4,P10 > P32 > P5✅5 > D10 > D32 > D5✅5 > 🪙❓ > P5❌_ > 🪙❗,5,5/166 [3.01%],90.090903,84.907579
5,P10 > P5✅5 > D10 > D5✅5 > 🪙❓ > P5❌_ > 🪙❗,5,5/166 [3.01%],84.361412,68.723738
6,P10 > P35 > P5✅7 > D10 > D35 > D5✅7 > D5❌_ > D10 > D35 > D5✅7 > 🪙❓ > 🪙❗,4,4/166 [2.41%],92.741985,103.960174
7,P10 > P32 > P5✅5 > D10 > D32 > D5✅5 > D5❌_ > D10 > D5❌2 > D5❌0 > D10 > D32 > D5✅5 > 🪙❓ > 🪙❗,4,4/166 [2.41%],88.470168,86.104928
8,P10 > P32 > P5✅5 > P5❌_ > P10 > P32 > P5✅5 > D10 > D32 > D5✅5 > 🪙❓ > 🪙❗,4,4/166 [2.41%],211.632436,170.632740
9,P10 > P35 > P5✅7 > D10 > D5✅5 > D5❌_ > D10 > D5✅5 > 🪙❓ > 🪙❗,4,4/166 [2.41%],84.124196,89.399873


In [38]:
#ACHTUNG- 2. und 3. Spaltung sind hier nur Eventmengen, keine Traces
events = pd.concat([
    logs["domestic"].assign(Log="domestic"),
    logs["int_without_trips"].assign(Log="international"),
], ignore_index=True)

events["Monat"] = pd.to_datetime(
    events["time:timestamp"], utc=True
).dt.strftime("%Y-%m")

trace_counts = (
    events.drop_duplicates(["Log", "case:concept:name", "Monat"])
    .groupby("Monat")
    .size()
    .rename("Traces insgesamt")
)

missing = events.loc[
    events["concept:name"].str.endswith("REJECTED by MISSING", na=False)
]

rejections = pd.crosstab(missing["Monat"], missing["concept:name"])

display(
    trace_counts.to_frame()
    .join(rejections)
    .fillna(0)
    .astype(int)
    .sort_index()
)

,Traces insgesamt,Declaration REJECTED by MISSING,Permit REJECTED by MISSING
Monat,,,
2017-01,278,14,0
2017-02,441,7,0
2017-03,495,0,0
2017-04,439,11,0
2017-05,557,19,5
2017-06,599,16,1
2017-07,497,24,12
2017-08,356,17,3
2017-09,561,40,4


In [39]:
log = logs["domestic"]

cases = (
    log.assign(Budget_Owner=log["org:role"].eq("BUDGET OWNER"))
    .groupby("case:concept:name")
    .agg(
        BudgetNumber=("case:BudgetNumber", "first"),
        Amount=("case:Amount", "first"),
        Budget_Owner=("Budget_Owner", "any"),
    )
)

cases["Budget_Owner"] = cases["Budget_Owner"].map({
    True: "beteiligt",
    False: "nicht beteiligt",
})

result = (
    cases.groupby(["BudgetNumber", "Budget_Owner"], dropna=False)
    .agg(
        Traces=("Amount", "size"),
        Mit_Betrag=("Amount", "count"),
        Min_Amount=("Amount", "min"),
        Max_Amount=("Amount", "max"),
    )
)

with pd.option_context("display.max_rows", None):
    display(result.style.format({
        "Min_Amount": "{:.2f}",
        "Max_Amount": "{:.2f}",
        "Spannweite": "{:.2f}",
    }, na_rep="—"))


log = logs["international"]
cases = (
    log.assign(Budget_Owner=log["org:role"].eq("BUDGET OWNER"))
    .groupby("case:concept:name")
    .agg(
        BudgetNumber=("case:BudgetNumber", "first"),
        Amount=("case:Amount", "first"),
        Budget_Owner=("Budget_Owner", "any"),
    )
)

cases["Budget_Owner"] = cases["Budget_Owner"].map({
    True: "beteiligt",
    False: "nicht beteiligt",
})

result = (
    cases.groupby(["BudgetNumber", "Budget_Owner"], dropna=False)
    .agg(
        Traces=("Amount", "size"),
        set_Betrag=("Amount", "count"),
        Min_Amount=("Amount", "min"),
        Max_Amount=("Amount", "max"),
    )
)

with pd.option_context("display.max_rows", None):
    display(result.style.format({
        "Min_Amount": "{:.2f}",
        "Max_Amount": "{:.2f}",
        "Spannweite": "{:.2f}",
    }, na_rep="—"))